# New York — Insurance Law (ISC / Insurance) → `data/new_york/ins_codes/*.md`

On **Justia**, the **New York Insurance Law** (McKinney’s consolidated **ISC**) is still published as the **2006 HTML tree** under **`/codes/new-york/2006/insurance/`** (article index pages like **`idx_isc0a1.html`** and section pages like **`isc0101_101.html`** for **§ 101**). The **2024 / 2025** “Insurance” hub pages link into this same tree — there are **no** modern **`/section-…`** URLs for this title on Justia.

**Important:** The **2006** snapshot may **not** reflect the latest amendments. The notebook records the **Justia** URL you fetched from; use the official site for current law.

**Cloudflare:** this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** BFS starting at **`…/2006/insurance/`**, following **only** links found in **`div.primary-content`** whose path stays under **`/codes/new-york/2006/insurance/`** and ends in **`.html`**. **Leaf** pages matching **`isc<digits>_<digits>.html`** (~**900+** sections) are downloaded.

**Files:** **`NY_INS_sec_<stem>.md`** where **`<stem>`** is the filename without **`.html`** (e.g. **`isc0101_101`**).

**Config:** **`MAX_SECTIONS`**, **`MAX_DISCOVERY_PAGES`**, **`REUSE_DISCOVERED_URLS`**, **`_ny_insurance_2006_section_urls.txt`**.

Run with **`ins_ipynb/`** as cwd, then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/new-york/2006/insurance"
START = f"{BASE}{PATH_PREFIX}/"

OUT_DIR = Path("data") / "new_york" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_ny_insurance_2006_section_urls.txt"
REUSE_DISCOVERED_URLS = True

SECTION_FILE_RE = re.compile(r"^isc\d+_\d+\.html$", re.I)


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def norm_url(absu: str) -> str:
    pr = urlparse(absu)
    return pr._replace(query="", fragment="").geturl().rstrip("/")


def stem_from_section_url(url: str) -> str:
    leaf = path_key(url).rsplit("/", 1)[-1]
    return leaf[:-5] if leaf.lower().endswith(".html") else leaf


def label_sort_key(stem: str) -> tuple:
    """Sort isc0101_101 by numeric pieces."""
    m = re.match(r"^isc(\d+)_(\d+)$", stem, re.I)
    if m:
        a, b = m.group(1), m.group(2)
        return (int(a), int(b))
    return (stem.lower(),)


def discover_section_urls() -> list[str]:
    from collections import deque

    start = START
    seen: set[str] = set()
    in_q: set[str] = {path_key(start).lower()}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    fetches = 0
    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url).lower()
        in_q.discard(pk)
        if pk in seen:
            continue
        if not pk.startswith(PATH_PREFIX):
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        soup = BeautifulSoup(html, "html.parser")
        pc = soup.select_one("div.primary-content")
        if not pc:
            continue
        for a in pc.find_all("a", href=True):
            nxt = norm_url(urljoin(url, a["href"]))
            p = path_key(nxt).lower()
            if not p.startswith(PATH_PREFIX):
                continue
            leaf = p.rsplit("/", 1)[-1]
            if not leaf.endswith(".html"):
                continue
            if SECTION_FILE_RE.match(leaf):
                sections.add(nxt)
                continue
            if p in seen or p in in_q:
                continue
            in_q.add(p)
            q.append(nxt)
    return sorted(sections, key=lambda u: label_sort_key(stem_from_section_url(u)))


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
        "There Is a Newer Version",
        "View Our Newest Version",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and ("N.Y. Laws" in s or "New York Code" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("Disclaimer:"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_insurance_law() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(stem_from_section_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section HTML pages under {PATH_PREFIX}")
        all_urls = sorted(found, key=lambda u: label_sort_key(stem_from_section_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        stem = stem_from_section_url(sec_url)
        dest = OUT_DIR / f"NY_INS_sec_{stem}.md"
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"New York Insurance Law ({stem})"
                md = (
                    f"# {title}\n\n"
                    f"**New York Consolidated Laws — Insurance Law (ISC)**\n\n"
                    f"**Source (Justia mirror, 2006 HTML tree):** {sec_url}\n\n"
                    f"**Verify current law:** [NYS Legislature — Laws of New York](https://www.nysenate.gov/legislation/laws/ISC)\n\n"
                    f"**Section file (Justia):** {stem}.html\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {stem}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_insurance_law()


Discovered 914 section HTML pages under /codes/new-york/2006/insurance
… 200/914 (wrote=200 skipped=0 failed=0)
… 400/914 (wrote=400 skipped=0 failed=0)
… 600/914 (wrote=600 skipped=0 failed=0)
… 800/914 (wrote=800 skipped=0 failed=0)
Done. wrote=914 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/new_york/ins_codes


{'wrote': 914, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
